# Forecasting

In [1]:
import pandas as pd

OUTPUT_FOLDER = "../Data/Generated/"

In [3]:
# load data sets
operations = pd.read_csv(
    OUTPUT_FOLDER +
    "fact_operations_stage2.csv"
)

financials = pd.read_csv(
    OUTPUT_FOLDER +
    "fact_financials.csv"
)

print(operations.shape)
print(financials.shape)

(56992, 50)
(56992, 56)


In [ ]:
# Converting dates dtype
operations["Date"] = pd.to_datetime(
    operations["Date"]
)

financials["Date"] = pd.to_datetime(
    financials["Date"]
)

In [5]:
# Aggregate daily operational KPIs
daily_ops = (
    operations
    .groupby("Date")
    .agg({
        "Passengers": "sum",
        "Seats_Offered": "sum",
        "Flights_Operated": "sum",
        "Delay_Minutes": "mean",
        "Demand_Index": "mean",
        "Security_Risk_Index": "mean",
        "Fuel_Index": "mean",
        "Competition_Index": "mean"
    })
    .reset_index()
)

daily_ops.head()

,Date,Passengers,Seats_Offered,Flights_Operated,Delay_Minutes,Demand_Index,Security_Risk_Index,Fuel_Index,Competition_Index
0,2020-01-01,7144.0,9258,48,11.423077,92.849793,2.0,101.99,100.00
1,2020-01-02,7228.0,9258,48,14.884615,93.124384,2.0,99.45,95.91
2,2020-01-03,7242.0,9258,48,14.461538,94.961198,2.0,102.59,100.00
3,2020-01-04,7072.0,9258,48,9.692308,93.064645,2.0,106.09,98.01
4,2020-01-05,7479.0,9258,48,3.615385,95.320227,2.0,99.06,100.00


In [6]:
# Create load factor
daily_ops["Load_Factor"] = (
    daily_ops["Passengers"]
    /
    daily_ops["Seats_Offered"]
)

In [7]:
# Daily delay rate
delay_rate = (
    operations
    .assign(
        Is_Delayed=(
            operations["Flight_Status"] == "Delayed"
        ).astype(int)
    )
    .groupby("Date")
    ["Is_Delayed"]
    .mean()
    .reset_index()
)

delay_rate.rename(
    columns={
        "Is_Delayed": "Delay_Rate"
    },
    inplace=True
)

delay_rate.head()

,Date,Delay_Rate
0,2020-01-01,0.307692
1,2020-01-02,0.269231
2,2020-01-03,0.192308
3,2020-01-04,0.192308
4,2020-01-05,0.115385


In [8]:
# Aggregate fanancial KPIs
daily_fin = (
    financials
    .groupby("Date")
    .agg({
        "Total_Revenue_USD": "sum",
        "Total_Operating_Cost_USD": "sum",
        "Operating_Profit_USD": "sum"
    })
    .reset_index()
)

daily_fin.head()

,Date,Total_Revenue_USD,Total_Operating_Cost_USD,Operating_Profit_USD
0,2020-01-01,1.697960e+06,1.234848e+06,463111.991591
1,2020-01-02,1.708711e+06,1.225411e+06,483300.681546
2,2020-01-03,1.751376e+06,1.245560e+06,505816.288096
3,2020-01-04,1.674300e+06,1.251738e+06,422562.347022
4,2020-01-05,1.809562e+06,1.239993e+06,569568.760618


In [9]:
# merging datasets
forecast = (
    daily_ops
    .merge(
        delay_rate,
        on="Date",
        how="left"
    )
    .merge(
        daily_fin,
        on="Date",
        how="left"
    )
)

forecast.head()

,Date,Passengers,Seats_Offered,Flights_Operated,Delay_Minutes,Demand_Index,Security_Risk_Index,Fuel_Index,Competition_Index,Load_Factor,Delay_Rate,Total_Revenue_USD,Total_Operating_Cost_USD,Operating_Profit_USD
0,2020-01-01,7144.0,9258,48,11.423077,92.849793,2.0,101.99,100.00,0.771657,0.307692,1.697960e+06,1.234848e+06,463111.991591
1,2020-01-02,7228.0,9258,48,14.884615,93.124384,2.0,99.45,95.91,0.780730,0.269231,1.708711e+06,1.225411e+06,483300.681546
2,2020-01-03,7242.0,9258,48,14.461538,94.961198,2.0,102.59,100.00,0.782242,0.192308,1.751376e+06,1.245560e+06,505816.288096
3,2020-01-04,7072.0,9258,48,9.692308,93.064645,2.0,106.09,98.01,0.763880,0.192308,1.674300e+06,1.251738e+06,422562.347022
4,2020-01-05,7479.0,9258,48,3.615385,95.320227,2.0,99.06,100.00,0.807842,0.115385,1.809562e+06,1.239993e+06,569568.760618


In [10]:
# Calandar features
forecast["Year"] = (
    forecast["Date"]
    .dt.year
)

forecast["Month"] = (
    forecast["Date"]
    .dt.month
)

forecast["Quarter"] = (
    forecast["Date"]
    .dt.quarter
)

forecast["Weekday"] = (
    forecast["Date"]
    .dt.dayofweek
)

forecast["Day_Of_Year"] = (
    forecast["Date"]
    .dt.dayofyear
)

In [11]:
# Seasonality flags
forecast["Is_Summer"] = (
    forecast["Month"]
    .isin([6,7,8])
    .astype(int)
)

forecast["Is_Shoulder_Season"] = (
    forecast["Month"]
    .isin([10,11])
    .astype(int)
)

forecast["Is_Winter"] = (
    forecast["Month"]
    .isin([12,1,2])
    .astype(int)
)

In [12]:
# Lag features 
# 1 day lag
forecast["Passengers_Lag_1"] = (
    forecast["Passengers"]
    .shift(1)
)

# 7 day lag
forecast["Passengers_Lag_7"] = (
    forecast["Passengers"]
    .shift(7)
)

# Revenue lag
forecast["Revenue_Lag_7"] = (
    forecast["Total_Revenue_USD"]
    .shift(7)
)

In [13]:
# Rolling passeger avg
forecast["Passengers_Rolling_7"] = (
    forecast["Passengers"]
    .rolling(7)
    .mean()
)

In [14]:
# Rolling revenue avg
forecast["Revenue_Rolling_7"] = (
    forecast["Total_Revenue_USD"]
    .rolling(7)
    .mean()
)

In [15]:
# removing nulls 
forecast = forecast.dropna()

In [16]:
#validation
print(
    "Rows:",
    len(forecast)
)

print(
    "Columns:",
    len(forecast.columns)
)

Rows: 2185
Columns: 27


In [17]:
forecast.columns.tolist()

['Date',
 'Passengers',
 'Seats_Offered',
 'Flights_Operated',
 'Delay_Minutes',
 'Demand_Index',
 'Security_Risk_Index',
 'Fuel_Index',
 'Competition_Index',
 'Load_Factor',
 'Delay_Rate',
 'Total_Revenue_USD',
 'Total_Operating_Cost_USD',
 'Operating_Profit_USD',
 'Year',
 'Month',
 'Quarter',
 'Weekday',
 'Day_Of_Year',
 'Is_Summer',
 'Is_Shoulder_Season',
 'Is_Winter',
 'Passengers_Lag_1',
 'Passengers_Lag_7',
 'Revenue_Lag_7',
 'Passengers_Rolling_7',
 'Revenue_Rolling_7']

In [18]:
forecast.head()

,Date,Passengers,Seats_Offered,Flights_Operated,Delay_Minutes,Demand_Index,Security_Risk_Index,Fuel_Index,Competition_Index,Load_Factor,...,Weekday,Day_Of_Year,Is_Summer,Is_Shoulder_Season,Is_Winter,Passengers_Lag_1,Passengers_Lag_7,Revenue_Lag_7,Passengers_Rolling_7,Revenue_Rolling_7
7,2020-01-08,7270.0,9258,48,12.230769,93.372797,2.0,103.07,100.00,0.785267,...,2,8,0,0,1,7441.0,7144.0,1.697960e+06,7268.428571,1.730870e+06
8,2020-01-09,7303.0,9258,48,17.615385,95.353324,2.0,98.12,100.00,0.788831,...,3,9,0,0,1,7270.0,7228.0,1.708711e+06,7279.142857,1.732822e+06
9,2020-01-10,7356.0,9258,48,17.807692,95.246835,2.0,102.17,99.43,0.794556,...,4,10,0,0,1,7303.0,7242.0,1.751376e+06,7295.428571,1.732600e+06
10,2020-01-11,6990.0,9258,48,6.846154,92.376846,2.0,98.15,100.00,0.755023,...,5,11,0,0,1,7356.0,7072.0,1.674300e+06,7283.714286,1.729503e+06
11,2020-01-12,7450.0,9258,48,16.000000,96.371342,2.0,98.14,96.12,0.804709,...,6,12,0,0,1,6990.0,7479.0,1.809562e+06,7279.571429,1.725833e+06


In [20]:
# Business validation:
print(
    "Total Revenue:",
    round(
        forecast["Total_Revenue_USD"].sum()/1_000_000,
        2
    ),
    "Million USD"
)

print(
    "Average Daily Passengers:",
    round(
        forecast["Passengers"].mean()
    )
)

print(
    "Average Daily Load Factor:",
    round(
        forecast["Load_Factor"].mean(),
        3
    )
)

Total Revenue: 4126.3 Million USD
Average Daily Passengers: 7669
Average Daily Load Factor: 0.795


In [21]:
forecast.to_csv(
    OUTPUT_FOLDER +
    "forecast_dataset.csv",
    index=False
)

print(
    "forecast_dataset.csv saved"
)

forecast_dataset.csv saved
